## 0. Setup

The PDF plumbing lives in `backend/pipeline/pdf_ingest.py` and gets imported rather than
retyped here — extracting clean text out of a CMS PDF is fiddly mechanical work and
there isn't much to learn from reading it twice. The parts that are actually worth
stepping through are written out inline below.

In [33]:
import json, re, hashlib, warnings, copy, sys, difflib
from pathlib import Path
from textwrap import shorten

warnings.filterwarnings("ignore")   # anaconda numexpr/bottleneck vs numpy 2.x noise

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND, DATA, PDFS = ROOT / "backend", ROOT / "backend" / "data", ROOT / "backend" / "data" / "pdfs"
CACHE = BACKEND / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BACKEND))

from pipeline.pdf_ingest import coverage_text, segment, locate, normalize

print("root:", ROOT)
print("\nsource PDFs:")
for p in sorted(PDFS.glob("*.pdf")):
    print(f"  {p.name:<56} {p.stat().st_size/1e6:>6.2f} MB")

root: /Users/nitinchaube/Studies/cotiviti-assessment

source PDFs:
  Article - Glucose Monitor - Policy Article (A52464).pdf    2.78 MB
  L33822_2020.pdf                                            0.45 MB
  L33822_2021.pdf                                            0.59 MB
  LCD - Glucose Monitors (L33822) superseded.pdf             0.56 MB
  LCD - Glucose Monitors (L33822).pdf                        0.54 MB
  adaStandardsOfCareSec7.pdf                                 0.70 MB


What you should see: six PDFs. Four versions of the LCD, the companion Policy
Article, and the ADA clinical guideline.

## 1. Load the PDFs without breaking character offsets

There's a subtle bug hiding here, so it's worth slowing down.

Every rule has to cite the sentence it came from. In the UI, clicking a rule highlights
that sentence in the document, and highlighting needs character offsets — "this rule
came from characters 6503 through 6604."

Now say you compute those offsets against the raw extraction, then clean the text up
afterward before displaying it. Collapse a couple of double spaces, strip a page footer,
and every offset past that point is now wrong. The highlight lands in the middle of a
different sentence and nobody notices until a reviewer asks why the citation looks like
garbage.

Normalize exactly once, then treat the result as the only text that exists. Offsets
index into it, the model reads it, the UI renders it. Nothing downstream ever touches
the raw bytes again.

CMS PDFs need three things handled, all in `pdf_ingest.py`:

- Every document opens with a multi-page contractor jurisdiction table that extracts as
  a long column of state names and has no policy content in it.
- Page headers and footers repeat across all twenty to sixty pages.
- The revision history at the end is a table whose columns interleave once flattened to
  text.

So we don't feed the whole document to anything. We cut it down to the coverage
criteria first.

In [34]:
V1_PDF = PDFS / "L33822_2020.pdf"          # effective 01/01/2020, ending 07/17/2021
V2_PDF = PDFS / "L33822_2021.pdf"          # effective 07/18/2021, ending 02/27/2022

v1_text = coverage_text(V1_PDF)
v2_text = coverage_text(V2_PDF)

import fitz
raw_pages = len(fitz.open(str(V1_PDF)))
raw_chars = len("".join(p.get_text() for p in fitz.open(str(V1_PDF))))

print(f"v1 raw PDF        : {raw_pages} pages, {raw_chars:,} chars")
print(f"v1 coverage text  : {len(v1_text):,} chars   <- what the pipeline actually sees")
print(f"v2 coverage text  : {len(v2_text):,} chars")
print()
print(f"we discarded {100*(1-len(v1_text)/raw_chars):.0f}% of the document as apparatus")
print("(contractor tables, evidence review, bibliography, revision history)")

v1 raw PDF        : 20 pages, 29,034 chars
v1 coverage text  : 12,710 chars   <- what the pipeline actually sees
v2 coverage text  : 12,639 chars

we discarded 56% of the document as apparatus
(contractor tables, evidence review, bibliography, revision history)


What you should see: the 20-page PDF reduces to roughly 12,700 characters of
actual coverage criteria — about 70% gets thrown away as apparatus. That's not
laziness. Feeding a model the contractor jurisdiction table burns tokens and invites it
to start inventing rules about the state of Delaware.

In [35]:
# This is the whole citation trail: if you can find a sentence, you can slice it
# back out exactly. Note we never ask the model for character offsets -- models are
# bad at counting characters and good at quoting text. The model quotes, we locate()
# it ourselves. A quote that can't be found is fabricated and the rule gets dropped.

probe = "The beneficiary has been using a BGM and performing frequent (four or more times a day) testing; and,"

span = locate(v1_text, probe)
print("probe :", probe)
print("span  :", span)
print("slice :", repr(v1_text[span[0]:span[1]]) if span else "NOT FOUND")
print()
ok = span and re.sub(r"\s+", " ", v1_text[span[0]:span[1]]).strip() == re.sub(r"\s+", " ", probe).strip()
print("round trip ok:", bool(ok))
assert ok, "OFFSET CONTRACT BROKEN - citations would point at the wrong text"

print()
print("same sentence in v2:", locate(v2_text, probe))
print("   ^ None. Medicare deleted this criterion on 07/18/2021.")
print("     That deletion is what this entire project exists to catch.")

probe : The beneficiary has been using a BGM and performing frequent (four or more times a day) testing; and,
span  : (6503, 6604)
slice : 'The beneficiary has been using a BGM and performing frequent (four or more times a day)\ntesting; and,'

round trip ok: True

same sentence in v2: None
   ^ None. Medicare deleted this criterion on 07/18/2021.
     That deletion is what this entire project exists to catch.


## 2. Segment, and look at what actually changed

Segmentation matters for two reasons. Extracting from a whole document at once gives
worse results than working section by section. And a policy isn't uniform anyway — the
coverage criteria section is dense with codifiable requirements, while the general
section is mostly documentation boilerplate.

In [4]:
v1_sections = segment(v1_text)
v2_sections = segment(v2_text)

print(f"{'section':<58} {'start':>7} {'end':>7} {'chars':>7}")
print("-" * 82)
for s in v1_sections:
    print(f"{s['name']:<58} {s['start']:>7} {s['end']:>7} {len(s['text']):>7}")

covered = sum(len(s["text"]) for s in v1_sections)
print("-" * 82)
assert covered == len(v1_text), "sections do not tile the document, offsets will drift"
print(f"tiles exactly: {covered} == {len(v1_text)}")

v1_cgm = next(s for s in v1_sections if s["name"].startswith("CONTINUOUS GLUCOSE"))
v2_cgm = next(s for s in v2_sections if s["name"].startswith("CONTINUOUS GLUCOSE"))
print(f"\nCGM section: v1 {len(v1_cgm['text'])} chars, v2 {len(v2_cgm['text'])} chars, "
      f"difference {len(v1_cgm['text']) - len(v2_cgm['text'])}")

section                                                      start     end   chars
----------------------------------------------------------------------------------
Coverage Indications, Limitations, and/or Medical Necessity       0    1468    1468
HOME BLOOD GLUCOSE MONITORS (BGM)                             1468    5975    4507
CONTINUOUS GLUCOSE MONITORS (CGM)                             5975    8773    2798
GENERAL                                                       8773   12710    3937
----------------------------------------------------------------------------------
tiles exactly: 12710 == 12710

CGM section: v1 2798 chars, v2 2706 chars, difference 92


In [5]:
# The actual diff between the two official versions -- this is the answer the
# detector below is supposed to arrive at on its own.
diff = difflib.unified_diff(
    v1_cgm["text"].split("\n"), v2_cgm["text"].split("\n"),
    fromfile="v1 (eff 01/01/2020)", tofile="v2 (eff 07/18/2021)", lineterm="", n=1)
for line in diff:
    print(line)

--- v1 (eff 01/01/2020)
+++ v2 (eff 07/18/2021)
@@ -4,36 +4,34 @@
 related Policy Article for additional information.
-Therapeutic CGMs and related supplies are covered by Medicare when all of the following
-coverage criteria (1-6) are met:
+Therapeutic CGMs and related supplies are covered by Medicare when all of the following coverage
+criteria (1-5) are met:
 1. The beneficiary has diabetes mellitus (Refer to the ICD-10 code list in the LCD-related Policy
 Article for applicable diagnoses); and,
-2. The beneficiary has been using a BGM and performing frequent (four or more times a day)
-testing; and,
-3. The beneficiary is insulin-treated with multiple (three or more) daily injections of insulin or a
-Medicare-covered continuous subcutaneous insulin infusion (CSII) pump; and,
-4. The beneficiary's insulin treatment regimen requires frequent adjustment by the beneficiary
-on the basis of BGM or CGM testing results; and,
-5. Within six (6) months prior to ordering the CGM, the treatin

## 3. Write one rule by hand

Before letting a model produce rules, write one yourself. Takes two minutes, and it
makes the schema concrete in a way that just reading a spec never does.

| Field | Why it exists |
|---|---|
| `applies_when` | Gates the rule. A CGM rule shouldn't fire on a wheelchair claim. |
| `requires` | The actual test. If it fails, `on_fail` happens. |
| `combinator` | `ALL`, `ANY`, or `N_OF_M`. Policy isn't always a plain conjunction. |
| `on_fail` | `DENY` or `REVIEW`, explicit, so polarity lives in the structure and not in how a sentence happens to be phrased. |
| `source_sentence` | The verbatim quote. Verified by search, never trusted. |
| `source_span` | Where we found it. This is what powers the UI highlight. |
| `codifiable` | `False` when the language is really a clinical judgment call. Those never auto-deny. |

In [6]:
hand_rule = {
    "rule_id": "CGM-002",
    "title": "Requires 4+ daily BGM testing",
    "summary": "The beneficiary must already be testing with a blood glucose monitor four or more times a day.",
    "source_doc": "L33822-2020",
    "source_sentence": "The beneficiary has been using a BGM and performing frequent (four or more times a day) testing; and,",
    "source_span": None,          # computed below, never typed by hand
    "logic": {
        "applies_when": [
            {"field": "procedure_code", "op": "in", "value": ["K0554", "K0553"]},
            {"field": "device_type",    "op": "==", "value": "therapeutic_cgm"},
        ],
        "requires": [
            {"field": "bgm_testing_freq_per_day", "op": ">=", "value": 4},
        ],
        "combinator": "ALL",
        "on_fail": "DENY",
    },
    "codifiable": True,
}

span = locate(v1_text, hand_rule["source_sentence"])
hand_rule["source_span"] = list(span) if span else None

print(json.dumps(hand_rule, indent=2))
print("\ncitation verifies:", span is not None)
print("text at span    :", repr(v1_text[span[0]:span[1]]))

{
  "rule_id": "CGM-002",
  "title": "Requires 4+ daily BGM testing",
  "summary": "The beneficiary must already be testing with a blood glucose monitor four or more times a day.",
  "source_doc": "L33822-2020",
  "source_sentence": "The beneficiary has been using a BGM and performing frequent (four or more times a day) testing; and,",
  "source_span": [
    6503,
    6604
  ],
  "logic": {
    "applies_when": [
      {
        "field": "procedure_code",
        "op": "in",
        "value": [
          "K0554",
          "K0553"
        ]
      },
      {
        "field": "device_type",
        "op": "==",
        "value": "therapeutic_cgm"
      }
    ],
    "requires": [
      {
        "field": "bgm_testing_freq_per_day",
        "op": ">=",
        "value": 4
      }
    ],
    "combinator": "ALL",
    "on_fail": "DENY"
  },
  "codifiable": true
}

citation verifies: True
text at span    : 'The beneficiary has been using a BGM and performing frequent (four or more times a day)\ntes

## 4. Compile and adjudicate, with no AI whatsoever

This is the engine. Pure Python, deterministic, and it's the part that would actually
run in production on ten million claims a day.

The separation here is a design decision I'd defend: a model authors rules, ordinary
code executes them. A denial has to be reproducible, identical every time, and
explainable three years later on appeal, and a model sitting in the decision path gives
you none of that.

### Why evaluation is three-valued

The obvious way to evaluate a condition is true or false. That's wrong here, and it's
wrong in a way that actually hurts people.

Say a rule needs `bgm_testing_freq_per_day` and the claim doesn't carry that field.
Two-valued logic says the condition is false, so the requirement fails, so deny. But we
didn't learn that the patient tests too rarely — we learned that we don't know.

Absence of data isn't evidence of ineligibility. So conditions return `TRUE`, `FALSE`,
or `UNKNOWN`, and `UNKNOWN` routes to a human instead of to a denial.

In [7]:
TRUE, FALSE, UNKNOWN = "TRUE", "FALSE", "UNKNOWN"
MISSING = object()

OPS = {
    "==": lambda a, b: a == b,   "!=": lambda a, b: a != b,
    ">=": lambda a, b: a >= b,   "<=": lambda a, b: a <= b,
    ">":  lambda a, b: a > b,    "<":  lambda a, b: a < b,
    "in": lambda a, b: a in b,   "not_in": lambda a, b: a not in b,
    "contains_any": lambda a, b: bool(set(a) & set(b)),
    "is_true":  lambda a, b: a is True,
    "is_false": lambda a, b: a is False,
}

def get_field(claim, field):
    if field in claim:
        return claim[field]
    if field in claim.get("attributes", {}):
        return claim["attributes"][field]
    return MISSING

def eval_condition(claim, cond):
    v = get_field(claim, cond["field"])
    if v is MISSING:
        return UNKNOWN
    try:
        return TRUE if OPS[cond["op"]](v, cond.get("value")) else FALSE
    except (TypeError, KeyError):
        return UNKNOWN                     # type mismatch is also "we cannot tell"

def combine(results, combinator="ALL", n=None):
    if combinator == "ALL":
        if FALSE in results: return FALSE          # one definite failure settles it
        return UNKNOWN if UNKNOWN in results else TRUE
    if combinator == "ANY":
        if TRUE in results:  return TRUE           # one definite success settles it
        return UNKNOWN if UNKNOWN in results else FALSE
    if combinator == "N_OF_M":
        t, u = results.count(TRUE), results.count(UNKNOWN)
        if t >= n: return TRUE
        return UNKNOWN if t + u >= n else FALSE
    raise ValueError(combinator)

print("truth table for ALL:")
for r in [[TRUE, TRUE], [TRUE, FALSE], [TRUE, UNKNOWN], [FALSE, UNKNOWN]]:
    print(f"  {str(r):<34} -> {combine(r, 'ALL')}")
print("\nrow 4 is the interesting one: a definite FALSE beats an UNKNOWN,")
print("because the claim fails regardless, so we can decide.")

truth table for ALL:
  ['TRUE', 'TRUE']                   -> TRUE
  ['TRUE', 'FALSE']                  -> FALSE
  ['TRUE', 'UNKNOWN']                -> UNKNOWN
  ['FALSE', 'UNKNOWN']               -> FALSE

row 4 is the interesting one: a definite FALSE beats an UNKNOWN,
because the claim fails regardless, so we can decide.


In [8]:
def eval_rule(claim, rule):
    lg, rid = rule["logic"], rule["rule_id"]

    applies = combine([eval_condition(claim, c) for c in lg.get("applies_when", [])], "ALL")
    if applies == FALSE:
        return {"rule_id": rid, "status": "NOT_APPLICABLE", "outcome": None}
    if applies == UNKNOWN:
        return {"rule_id": rid, "status": "APPLICABILITY_UNKNOWN", "outcome": "REVIEW",
                "why": "cannot tell whether this rule applies", "citation": rule.get("source_sentence")}

    # A non-codifiable rule encodes a clinical judgment. It applies, but it is never
    # allowed to decide on its own.
    if not rule.get("codifiable", True):
        return {"rule_id": rid, "status": "NOT_CODIFIABLE", "outcome": "REVIEW",
                "why": "policy language requires human judgment", "citation": rule.get("source_sentence")}

    per = [eval_condition(claim, c) for c in lg["requires"]]
    verdict = combine(per, lg.get("combinator", "ALL"), lg.get("n"))

    if verdict == TRUE:
        return {"rule_id": rid, "status": "PASS", "outcome": None}
    if verdict == FALSE:
        failed = [c["field"] for c, r in zip(lg["requires"], per) if r == FALSE]
        return {"rule_id": rid, "status": "FAIL", "outcome": lg["on_fail"],
                "why": f"failed on {failed}", "citation": rule.get("source_sentence")}
    unknown = [c["field"] for c, r in zip(lg["requires"], per) if r == UNKNOWN]
    return {"rule_id": rid, "status": "UNDETERMINED", "outcome": "REVIEW",
            "why": f"missing data: {unknown}", "citation": rule.get("source_sentence")}


PRECEDENCE = {"DENY": 3, "REVIEW": 2, "PAY": 1}

def adjudicate(claim, rules):
    trace = [eval_rule(claim, r) for r in rules]
    outcomes = [t["outcome"] for t in trace if t["outcome"]]
    decision = "PAY" if not outcomes else max(outcomes, key=lambda o: PRECEDENCE[o])
    return {"claim_id": claim["claim_id"], "decision": decision,
            "fired": [t for t in trace if t["status"] not in ("NOT_APPLICABLE", "PASS")],
            "trace": trace}

print("engine defined. no model was consulted.")

engine defined. no model was consulted.


In [9]:
# claim loading validates too -- malformed records get rejected with a reason
REQUIRED = ["claim_id", "date_of_service", "procedure_code", "units", "billed_amount"]

def load_claims(path):
    blob = json.loads(Path(path).read_text())["claims"]
    good, rejected, seen = [], [], set()
    for c in blob:
        missing = [f for f in REQUIRED if f not in c]
        if missing:
            rejected.append((c.get("claim_id", "?"), f"missing required field(s): {missing}")); continue
        if c["claim_id"] in seen:
            rejected.append((c["claim_id"], "duplicate claim_id")); continue
        if not isinstance(c["units"], int) or c["units"] < 1:
            rejected.append((c["claim_id"], f"units must be an integer >= 1, got {c['units']}")); continue
        seen.add(c["claim_id"]); good.append(c)
    return good, rejected

claims, rejected = load_claims(DATA / "claims.json")
by_id = {c["claim_id"]: c for c in claims}
print(f"accepted: {len(claims)}   rejected: {len(rejected)}")
for cid, why in rejected:
    print(f"  {cid}: {why}")
assert len(claims) == 20 and len(rejected) == 3

accepted: 20   rejected: 3
  M-001: units must be an integer >= 1, got 0
  C-001: duplicate claim_id
  M-003: missing required field(s): ['procedure_code']


### Run four claims through the one hand-written rule

Four cases, each one hitting a different branch.

In [10]:
def show(claim_id, rules):
    claim = by_id[claim_id]
    res = adjudicate(claim, rules)
    freq = claim["attributes"].get("bgm_testing_freq_per_day", "ABSENT")
    print(f"{claim_id}  procedure={claim['procedure_code']:<6} bgm_testing/day={freq}")
    print(f"  decision: {res['decision']}")
    for t in res["trace"]:
        print(f"    {t['rule_id']}: {t['status']}  {t.get('why','')}")
        if t.get("citation"):
            print(f"      because: \"{shorten(t['citation'], 66)}\"")
    if not res["fired"]:
        print("    -> nothing objected, so PAY")
    print()

print("=" * 78)
print("ONE rule loaded (the hand-written CGM-002). Watch each branch.")
print("=" * 78)
show("C-001", [hand_rule])   # tests 5x/day  -> passes
show("C-007", [hand_rule])   # tests 1x/day  -> fails -> DENY
show("C-008", [hand_rule])   # field absent  -> UNKNOWN -> REVIEW
show("C-014", [hand_rule])   # respiratory device -> rule does not apply -> PAY

ONE rule loaded (the hand-written CGM-002). Watch each branch.
C-001  procedure=K0554  bgm_testing/day=5
  decision: PAY
    CGM-002: PASS  
    -> nothing objected, so PAY

C-007  procedure=K0554  bgm_testing/day=1
  decision: DENY
    CGM-002: FAIL  failed on ['bgm_testing_freq_per_day']
      because: "The beneficiary has been using a BGM and performing frequent [...]"

C-008  procedure=K0554  bgm_testing/day=ABSENT
  decision: REVIEW
    CGM-002: UNDETERMINED  missing data: ['bgm_testing_freq_per_day']
      because: "The beneficiary has been using a BGM and performing frequent [...]"

C-014  procedure=E0470  bgm_testing/day=0
  decision: PAY
    CGM-002: NOT_APPLICABLE  
    -> nothing objected, so PAY



What to notice:

- `C-001` tests 5 times a day, passes, `PAY`.
- `C-007` tests once a day, fails, `DENY`. Hang onto this claim — Medicare deleted this
  criterion on 07/18/2021, so this denial is actually wrong, and section 8 catches it.
- `C-008` is missing the field, so `REVIEW`, not `DENY`. Absence of data doing its job.
- `C-014` is a respiratory device. Nothing applies, nothing objects, `PAY`. No rule
  means no basis to deny.

In [11]:
# the app ships this same engine as backend/pipeline/engine.py -- check they agree
from pipeline.engine import adjudicate as lib_adjudicate

reference = json.loads((DATA / "reference_ruleset.json").read_text())["rules"]
for r in reference:
    sp = locate(v1_text, r["source_sentence"])
    r["source_span"], r["source_doc"] = (list(sp) if sp else None), "L33822-2020"
    assert sp, f"reference rule {r['rule_id']} cites text not present in v1"

mismatch = [c["claim_id"] for c in claims
            if adjudicate(c, reference)["decision"] != lib_adjudicate(c, reference)["decision"]]
print(f"reference ruleset: {len(reference)} rules, all citations verified against the real PDF")
print(f"notebook engine vs pipeline/engine.py: {len(mismatch)} disagreements", mismatch or "")
assert not mismatch

reference ruleset: 12 rules, all citations verified against the real PDF


notebook engine vs pipeline/engine.py: 0 disagreements 


---

### End of the AI-free half

At this point you have a working claim adjudication engine that has never once called
a model, running on rules whose every citation has been checked against a real CMS PDF.

Everything the language model does from here on is authoring. It reads English and
proposes rules in the schema you just hand-wrote yourself. It never touches a claim
decision.

## 5. The model finally shows up

Its job is narrow: read a section of policy prose and emit rules in the exact schema
you hand-wrote in section 3. It authors. It doesn't decide anything.

Three things keep it reliable enough to actually build on.

**Strict JSON schema.** We don't ask for JSON and hope for the best. The API gets a
schema and is constrained to conform to it, which removes a whole category of failure
before it can even happen.

**No offsets requested.** The model quotes, we locate. A quote we can't find is fake.

**A closed field list.** The model only sees the claim fields that actually exist.
Anything it invents on top of that gets caught in section 6.

### One schema wrinkle

Strict JSON schema mode has no union types, so a `value` that could be a string,
number, boolean, or list isn't directly expressible. The workaround is typed slots
(`value_string`, `value_number`, ...) plus a `value_kind` discriminator, collapsed back
down in Python afterward. A little ugly, but entirely mechanical.

In [12]:
import os
from dotenv import load_dotenv

load_dotenv(BACKEND / ".env")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL   = os.getenv("OPENAI_MODEL", "gpt-4.1")

print("model      :", MODEL)
print("api key    :", "present" if API_KEY else "NOT SET")
print("cached msgs:", len(list(CACHE.glob("*.json"))))
if not API_KEY:
    print("\nNo key. Cached responses are still used if present; cells needing a live")
    print("call will say so and fall back. Copy backend/.env.example to backend/.env.")

model      : gpt-4.1
api key    : present
cached msgs: 153


In [13]:
class NoAPIKey(Exception):
    pass

def chat_json(system, user, schema, schema_name, temperature=0.0, run_tag=""):
    """Strict-schema model call, with every response cached to disk.

    The cache key covers prompt, schema, model and run_tag, so re-running this notebook
    costs nothing and works offline. run_tag is what lets section 7 make five genuinely
    separate calls instead of hitting one cached answer five times.
    """
    key = hashlib.sha256(json.dumps(
        [system, user, schema, MODEL, temperature, run_tag], sort_keys=True).encode()).hexdigest()[:20]
    path = CACHE / f"{schema_name}_{key}.json"
    if path.exists():
        return json.loads(path.read_text())["parsed"], "cache"
    if not API_KEY:
        raise NoAPIKey("no cached response for this prompt and no API key set")

    from openai import OpenAI
    resp = OpenAI(api_key=API_KEY).chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": schema_name, "strict": True, "schema": schema}})
    raw = resp.choices[0].message.content
    parsed = json.loads(raw)
    path.write_text(json.dumps({"parsed": parsed, "raw": raw, "model": MODEL,
                                "temperature": temperature, "run_tag": run_tag,
                                "usage": {"prompt": resp.usage.prompt_tokens,
                                          "completion": resp.usage.completion_tokens}}, indent=2))
    return parsed, "api"

print("chat_json ready. responses land in backend/cache/.")

chat_json ready. responses land in backend/cache/.


In [14]:
schema_blob    = json.loads((DATA / "claim_schema.json").read_text())
ALLOWED_FIELDS = sorted(list(schema_blob["top_level"]) + list(schema_blob["attributes"]))

CONDITION = {
    "type": "object",
    "properties": {
        "field":         {"type": "string"},
        "op":            {"type": "string", "enum": list(OPS)},
        "value_kind":    {"type": "string", "enum": ["none", "string", "number", "boolean", "string_list"]},
        "value_string":  {"type": ["string", "null"]},
        "value_number":  {"type": ["number", "null"]},
        "value_boolean": {"type": ["boolean", "null"]},
        "value_list":    {"type": ["array", "null"], "items": {"type": "string"}},
    },
    "required": ["field", "op", "value_kind", "value_string", "value_number", "value_boolean", "value_list"],
    "additionalProperties": False,
}

EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {"rules": {"type": "array", "items": {
        "type": "object",
        "properties": {
            "rule_id": {"type": "string"}, "title": {"type": "string"},
            "summary": {"type": "string"}, "source_sentence": {"type": "string"},
            "codifiable": {"type": "boolean"},
            "logic": {"type": "object", "properties": {
                "applies_when": {"type": "array", "items": CONDITION},
                "requires":     {"type": "array", "items": CONDITION},
                "combinator":   {"type": "string", "enum": ["ALL", "ANY", "N_OF_M"]},
                "n":            {"type": ["integer", "null"]},
                "on_fail":      {"type": "string", "enum": ["DENY", "REVIEW"]}},
                "required": ["applies_when", "requires", "combinator", "n", "on_fail"],
                "additionalProperties": False}},
        "required": ["rule_id", "title", "summary", "source_sentence", "codifiable", "logic"],
        "additionalProperties": False}}},
    "required": ["rules"], "additionalProperties": False,
}

def coerce_condition(c):
    return {"field": c["field"], "op": c["op"], "value": {
        "none": None, "string": c["value_string"], "number": c["value_number"],
        "boolean": c["value_boolean"], "string_list": c["value_list"]}[c["value_kind"]]}

def coerce_rule(r, doc_text, doc_id):
    lg, out = r["logic"], dict(r)
    out["logic"] = {"applies_when": [coerce_condition(c) for c in lg["applies_when"]],
                    "requires":     [coerce_condition(c) for c in lg["requires"]],
                    "combinator":   lg["combinator"], "n": lg.get("n"), "on_fail": lg["on_fail"]}
    sp = locate(doc_text, r["source_sentence"])
    out["source_span"], out["source_doc"] = (list(sp) if sp else None), doc_id
    return out

# bare field names weren't enough -- first run it emitted device_type == "cgm" and
# got rejected for a value it was never told was illegal. give it types + enums too.
_KNOWN_CODES = sorted(k for k in schema_blob["codes"] if not k.startswith("_"))

def _catalogue_line(field, ftype, enums, codes):
    line = f"   - {field} ({ftype}"
    if field in enums:
        line += ", one of: " + ", ".join(enums[field])
    elif field == "procedure_code":
        line += ", HCPCS codes appearing in this policy: " + ", ".join(codes)
    return line + ")"

FIELD_CATALOGUE = "\n".join(
    _catalogue_line(f, {**schema_blob["top_level"], **schema_blob["attributes"]}[f],
                    schema_blob.get("enums", {}), _KNOWN_CODES)
    for f in ALLOWED_FIELDS)

print(f"{len(ALLOWED_FIELDS)} claim fields the model may reference:")
print(FIELD_CATALOGUE)

21 claim fields the model may reference:
   - bgm_testing_freq_per_day (int)
   - billed_amount (number)
   - claim_id (string)
   - csii_pump (bool)
   - csii_pump_medicare_covered (bool)
   - date_of_service (date)
   - device_type (string, one of: therapeutic_cgm, home_bgm, bgm_special_features, other)
   - diabetes_diagnosis (bool)
   - diagnosis_codes (list[string])
   - followup_visit_within_6mo (bool)
   - in_person_visit_within_6mo (bool)
   - insulin_administrations_per_day (int)
   - insulin_injections_per_day (int)
   - insulin_treated (bool)
   - manual_dexterity_impairment (bool)
   - procedure_code (string, HCPCS codes appearing in this policy: E0470, E0607, E2100, E2101, K0553, K0554)
   - regimen_requires_frequent_adjustment (bool)
   - severe_visual_impairment (bool)
   - sufficient_training_documented (bool)
   - swo_on_file (bool)
   - units (int)


In [15]:
EXTRACT_SYSTEM = f"""You convert health insurance coverage policy text into executable claim rules.

You will be given one section of a Medicare Local Coverage Determination. Emit one rule
for each distinct, checkable requirement or limitation the text states.

HARD CONSTRAINTS

1. source_sentence must be copied VERBATIM from the supplied text. Do not paraphrase, do
   not merge sentences, do not fix typos or drop trailing "; and,". It is checked by exact
   search and the rule is discarded if it is not found.

2. Every `field` must come from this list and nothing else. The type is given, and where a
   field has a fixed set of legal values those are the ONLY values you may use for it:
{FIELD_CATALOGUE}

3. applies_when gates the rule. A rule about continuous glucose monitors must not fire on
   unrelated equipment. Use procedure_code and/or device_type.

4. requires holds the actual test. on_fail is DENY when the policy states a coverage
   condition or an outright exclusion, REVIEW when claim data cannot settle it.

5. Set codifiable=false when the requirement turns on clinical judgment that a claim
   record cannot settle (phrases like "severe enough to require", "reasonable and
   necessary", "as documented in the medical record"). Those route to a human and never
   auto-deny, so use on_fail=REVIEW.

6. For an unconditional exclusion, gate it with applies_when and give it a requirement
   that can never be met, so it always denies when it applies.

7. Emit nothing for background, evidence review, or boilerplate. Zero rules is a valid
   answer for a section stating no checkable requirement.

8. Emit exactly ONE rule per numbered coverage criterion. Do not emit an aggregate rule
   restating that all criteria must be met. That is already implied by the individual
   rules, and encoding it again double-counts every requirement.

9. Never reference an identifier field (claim_id, date_of_service) inside `requires`.
   Those identify a claim, they do not test it.

10. Operators must match the field's type. contains_any is only for list fields. The
    comparison operators are only for numeric fields. is_true and is_false are only for
    boolean fields. For `in`, every value must be one the field can actually hold.

11. If a requirement cannot be expressed with the available fields, omit the rule. Do not
    approximate it with an unrelated field."""


def extract_rules(section_text, doc_text, doc_id, run_tag="", temperature=0.0):
    user = f"Extract rules from this policy section.\n\n---\n{section_text}\n---"
    parsed, src = chat_json(EXTRACT_SYSTEM, user, EXTRACTION_SCHEMA, "rules",
                            temperature=temperature, run_tag=run_tag)
    return [coerce_rule(r, doc_text, doc_id) for r in parsed["rules"]], src


LLM_AVAILABLE, extracted = True, []
try:
    extracted, src = extract_rules(v1_cgm["text"], v1_text, "L33822-2020", run_tag="run0")
    print(f"source: {src}   rules returned: {len(extracted)}\n")
    for r in extracted:
        print(f"  [{'ok ' if r['source_span'] else 'BAD'}] {r['rule_id']:<10} {r['title']}")
        print(f"        requires={r['logic']['requires']}")
        print(f"        cites   \"{shorten(r['source_sentence'], 64)}\"")
except NoAPIKey as e:
    LLM_AVAILABLE = False
    print("SKIPPED:", e)
    print("\nSections 5, 7 and 8 need a key or a warm cache. Everything below still runs")
    print("on the hand-authored reference ruleset, and says so wherever it does.")

source: cache   rules returned: 7

  [ok ] cgm-1-diabetes-diagnosis Diabetes diagnosis required for CGM coverage
        requires=[{'field': 'diabetes_diagnosis', 'op': 'is_true', 'value': True}]
        cites   "1. The beneficiary has diabetes mellitus (Refer to the [...]"
  [ok ] cgm-2-bgm-testing-freq Frequent BGM testing required for CGM coverage
        requires=[{'field': 'bgm_testing_freq_per_day', 'op': '>=', 'value': 4}]
        cites   "2. The beneficiary has been using a BGM and performing [...]"
  [ok ] cgm-3-insulin-treated Insulin-treated with multiple daily injections or Medicare-covered CSII pump required for CGM coverage
        requires=[{'field': 'insulin_injections_per_day', 'op': '>=', 'value': 3}, {'field': 'insulin_treated', 'op': 'is_true', 'value': True}]
        cites   "3. The beneficiary is insulin-treated with multiple (three [...]"
  [ok ] cgm-4-regimen-adjustment Insulin regimen requires frequent adjustment for CGM coverage
        requires=[{'field': 're

## 6. The validation gauntlet

Everything the model produced up to this point is a proposal, not a fact. Four cheap
checks catch most bad output, and none of them cost another model call.

| Check | Catches |
|---|---|
| Citation findable in the source | fabricated quotes |
| Every field in the allowed list | invented claim attributes |
| Every operator implemented | plausible-looking nonsense |
| Operator matches the field's declared type | `claim_id contains_any ['K0554']` |
| Values are ones the field can hold | `device_type in ['PDAC_approved']` |
| No identifier field inside `requires` | rules that test a claim's name |
| `requires` non-empty | a rule that can never fire |

This is the section people tend to skip, which is also why their demo will confidently
deny a claim based on a sentence that was never in the document.

The last three checks in that table got added after the first real run, which is worth
admitting. The original gauntlet only asked whether a field existed. The extractor
promptly produced `claim_id contains_any ['K0554','K0553']` and
`device_type in ['PDAC_approved']`. Both reference real fields. Both are meaningless.
Both sailed straight through, reached the engine, and turned twelve claims into manual
review.

The fix wasn't a smarter model. The claim schema already declares that `claim_id` is a
string and that `device_type` has four legal values. Type-checking the operators and
range-checking the values against that declaration catches both problems,
deterministically, for free. A schema you already wrote is about the cheapest validator
you'll ever get.

In [16]:
FIELD_TYPES = {**schema_blob["top_level"], **schema_blob["attributes"]}
FIELD_ENUMS = schema_blob.get("enums", {})
IDENTIFIER_FIELDS = {"claim_id", "date_of_service"}

LIST_OPS = {"contains_any"}
NUM_OPS  = {">=", "<=", ">", "<"}
BOOL_OPS = {"is_true", "is_false"}


def validate_rule(rule, doc_text, allowed_fields):
    problems = []
    span = locate(doc_text, rule["source_sentence"])
    if span is None:
        problems.append("citation not found in source document (fabricated quote)")

    for where, conds in (("applies_when", rule["logic"]["applies_when"]),
                         ("requires", rule["logic"]["requires"])):
        for c in conds:
            f, op, val = c["field"], c["op"], c.get("value")

            if f not in allowed_fields:
                problems.append(f"unknown field '{f}' (not in claim schema)")
                continue
            if op not in OPS:
                problems.append(f"unsupported operator '{op}'")
                continue

            # identifiers name a claim, they don't test one
            if where == "requires" and f in IDENTIFIER_FIELDS:
                problems.append(f"'{f}' is an identifier and cannot be a requirement")

            # operator has to match the field's declared type
            ftype = FIELD_TYPES[f]
            if op in LIST_OPS and not ftype.startswith("list"):
                problems.append(f"operator '{op}' needs a list field, but '{f}' is {ftype}")
            if op in NUM_OPS and ftype not in ("int", "number"):
                problems.append(f"operator '{op}' needs a numeric field, but '{f}' is {ftype}")
            if op in BOOL_OPS and ftype != "bool":
                problems.append(f"operator '{op}' needs a boolean field, but '{f}' is {ftype}")

            # value has to be one the field can actually hold
            if f in FIELD_ENUMS:
                supplied = val if isinstance(val, list) else [val]
                bad = [v for v in supplied if v is not None and v not in FIELD_ENUMS[f]]
                if bad:
                    problems.append(f"value(s) {bad} are not valid for '{f}' "
                                    f"(allowed: {FIELD_ENUMS[f]})")

    if not rule["logic"]["requires"]:
        problems.append("rule has no requirements, it can never fire")
    if rule["logic"].get("combinator") == "N_OF_M" and not rule["logic"].get("n"):
        problems.append("N_OF_M combinator without n")
    return span, problems


# Two deliberately broken rules, so you watch the gauntlet actually reject something.
poisoned = [
    {"rule_id": "BAD-001", "title": "Fabricated citation",
     "source_sentence": "The beneficiary must demonstrate a hemoglobin A1c below 7.0 percent.",
     "logic": {"applies_when": [{"field": "procedure_code", "op": "in", "value": ["K0554"]}],
               "requires": [{"field": "bgm_testing_freq_per_day", "op": ">=", "value": 4}],
               "combinator": "ALL", "on_fail": "DENY"}, "codifiable": True},
    {"rule_id": "BAD-002", "title": "Invented claim field",
     "source_sentence": "The beneficiary has diabetes mellitus (Refer to the ICD-10 code list in the LCD-related Policy Article for applicable diagnoses); and,",
     "logic": {"applies_when": [{"field": "procedure_code", "op": "in", "value": ["K0554"]}],
               "requires": [{"field": "patient_motivation_score", "op": ">=", "value": 7}],
               "combinator": "ALL", "on_fail": "DENY"}, "codifiable": True},
]

print("gauntlet on 2 deliberately broken rules:\n")
for r in poisoned:
    _, problems = validate_rule(r, v1_text, ALLOWED_FIELDS)
    print(f"  {r['rule_id']} ({r['title']})")
    for p in problems:
        print(f"      REJECTED: {p}")
    print()
assert validate_rule(poisoned[0], v1_text, ALLOWED_FIELDS)[1], "fabricated citation slipped through"
assert validate_rule(poisoned[1], v1_text, ALLOWED_FIELDS)[1], "invented field slipped through"
print("both rejected as expected")

gauntlet on 2 deliberately broken rules:

  BAD-001 (Fabricated citation)
      REJECTED: citation not found in source document (fabricated quote)

  BAD-002 (Invented claim field)
      REJECTED: unknown field 'patient_motivation_score' (not in claim schema)

both rejected as expected


In [17]:
if LLM_AVAILABLE and extracted:
    clean, dirty = [], []
    for r in extracted:
        _, problems = validate_rule(r, v1_text, ALLOWED_FIELDS)
        (clean if not problems else dirty).append((r, problems))
    print(f"model rules passing validation: {len(clean)}/{len(extracted)}")
    for r, ps in dirty:
        print(f"  rejected {r['rule_id']}: {ps}")
    working_rules, RULES_ARE_MODEL_OUTPUT = [r for r, _ in clean], True
else:
    print("no model output available, using the hand-authored reference ruleset")
    working_rules, RULES_ARE_MODEL_OUTPUT = reference, False

for r in reference:
    assert not validate_rule(r, v1_text, ALLOWED_FIELDS)[1]
print(f"\nreference ruleset: {len(reference)} rules, all valid")
print(f"working ruleset  : {len(working_rules)} rules "
      f"({'model-extracted' if RULES_ARE_MODEL_OUTPUT else 'hand-authored fallback'})")

model rules passing validation: 7/7

reference ruleset: 12 rules, all valid
working ruleset  : 7 rules (model-extracted)


## 7. Does it give the same answer twice?

A demo that runs extraction once and shows you the output tells you nothing about
whether it actually works. Run it five times and you learn something real.

**Stability.** Five independent calls at temperature 0. Any rule that shows up in fewer
than five runs gets flagged, since a rule the extractor isn't sure about isn't one you
want to deploy.

**Accuracy against the hand-authored ruleset.** `reference_ruleset.json` was written by
reading the policy by hand, so we can report coverage, logic exactness, and spurious
output against it. Matching is done on the cited sentence, which is a fair proxy —
two rules citing the same sentence are attempts at the same requirement.

In [18]:
def norm_conditions(cs):
    """Canonical, order-insensitive form of a condition list.

    is_true and is_false ignore their operand entirely, so the gold file writing
    value=null and the model writing value=true are the same condition. Comparing them
    raw made every rule look 'encoded differently', which was a bug in the metric rather
    than a fault in the model.
    """
    out = []
    for c in cs:
        val = None if c["op"] in ("is_true", "is_false") else c.get("value")
        out.append(f"{c['field']}|{c['op']}|{json.dumps(val, sort_keys=True)}")
    return sorted(out)


def requires_hash(rule):
    return hashlib.sha256(json.dumps(
        {"r": norm_conditions(rule["logic"]["requires"]),
         "c": rule["logic"]["combinator"]}, sort_keys=True).encode()).hexdigest()[:12]


def logic_hash(rule):
    lg = rule["logic"]
    return hashlib.sha256(json.dumps(
        {"a": norm_conditions(lg["applies_when"]), "r": norm_conditions(lg["requires"]),
         "c": lg["combinator"], "n": lg.get("n"), "f": lg["on_fail"]},
        sort_keys=True).encode()).hexdigest()[:12]

# a quoted criterion may carry its list marker ("2. The beneficiary has been using
# a BGM..."), which is formatting, not part of the requirement -- left in, it made the
# same rule look like two different rules across runs. stripped for matching only;
# source_sentence stays verbatim so it can still be located.
_LIST_MARKER = re.compile(r"^\s*(?:\(?\d{1,2}[.)]|\(?[a-z][.)])\s+")

def cite_key(rule):
    s = re.sub(r"\s+", " ", rule["source_sentence"]).strip().lower()
    return _LIST_MARKER.sub("", s).strip()


N_RUNS, runs = 5, []
if LLM_AVAILABLE:
    for i in range(N_RUNS):
        try:
            rs, src = extract_rules(v1_cgm["text"], v1_text, "L33822-2020", run_tag=f"stability{i}")
            rs = [r for r in rs if not validate_rule(r, v1_text, ALLOWED_FIELDS)[1]]
            runs.append(rs)
            print(f"  run {i}: {len(rs)} valid rules   ({src})")
        except NoAPIKey:
            break

if len(runs) == N_RUNS:
    from collections import Counter
    seen = Counter()
    for rs in runs:
        for k in {cite_key(r) for r in rs}:
            seen[k] += 1
    print(f"\n{'agreement':<12} cited sentence")
    print("-" * 88)
    for k, n in seen.most_common():
        print(f"{n}/{N_RUNS:<10} {shorten(k, 66)}{'' if n == N_RUNS else '   <-- unstable'}")
    unanimous = sum(1 for n in seen.values() if n == N_RUNS)
    print(f"\nunanimous across all {N_RUNS} runs: {unanimous}/{len(seen)} distinct rules")
    STABILITY = (unanimous, len(seen))
else:
    print("stability needs 5 live or cached runs. skipped.")
    STABILITY = None

  run 0: 9 valid rules   (cache)
  run 1: 8 valid rules   (cache)
  run 2: 8 valid rules   (cache)
  run 3: 8 valid rules   (cache)
  run 4: 8 valid rules   (cache)

agreement    cited sentence
----------------------------------------------------------------------------------------
5/5          the beneficiary's insulin treatment regimen requires [...]
5/5          the beneficiary is insulin-treated with multiple (three or [...]
5/5          the beneficiary has been using a bgm and performing frequent [...]
5/5          the beneficiary has diabetes mellitus (refer to the icd-10 [...]
5/5          within six (6) months prior to ordering the cgm, the [...]
5/5          every six (6) months following the initial prescription of [...]
5/5          claims for a bgm and related supplies, billed in addition to [...]
4/5          only one (1) uos of code k0553 may be billed to the dme macs [...]   <-- unstable
1/5          billing more than 1 uos per 30 days of code k0553 will be [...]   <-- u

In [19]:
gold_by_cite = {cite_key(r): r for r in reference}

# only the CGM section was extracted, so score against the gold rules that live
# in it, not the whole document -- wrong denominator is worse than no score.
in_scope = {k: r for k, r in gold_by_cite.items() if locate(v1_cgm["text"], r["source_sentence"])}
print(f"gold rules in the extracted section: {len(in_scope)} of {len(gold_by_cite)} document-wide\n")

if RULES_ARE_MODEL_OUTPUT:
    got_by_cite = {cite_key(r): r for r in working_rules}

    # match on the cited sentence first, then give leftovers a second chance on
    # identical logic -- a paragraph can state one requirement twice, and the model
    # may quote the other sentence than gold did. counting that as a miss AND a
    # spurious is the metric being wrong twice about the same thing.
    matched, used = {}, set()
    for k in in_scope:
        if k in got_by_cite:
            matched[k], _ = got_by_cite[k], used.add(k)
    for k, gold_rule in in_scope.items():
        if k in matched:
            continue
        gh = requires_hash(gold_rule)
        for gk, got_rule in got_by_cite.items():
            if gk not in used and requires_hash(got_rule) == gh:
                matched[k] = got_rule; used.add(gk)
                print(f"  matched by identical logic despite a different citation:")
                print(f"     gold  cites \"{shorten(k, 62)}\"")
                print(f"     model cites \"{shorten(gk, 62)}\"")
                break

    found    = sorted(matched)
    missed   = [k for k in in_scope if k not in matched]
    spurious = [k for k in got_by_cite if k not in gold_by_cite and k not in used]
    same_req = [k for k in found if requires_hash(matched[k]) == requires_hash(in_scope[k])]
    same_all = [k for k in found if logic_hash(matched[k]) == logic_hash(in_scope[k])]
    got_by_cite = {**got_by_cite, **{k: matched[k] for k in found}}
    print()

    print(f"found by extractor : {len(found)}/{len(in_scope)}")
    print(f"  same requirements: {len(same_req)}")
    print(f"  fully identical  : {len(same_all)}  (also matching applies_when and on_fail)")
    print(f"missed             : {len(missed)}")
    print(f"spurious           : {len(spurious)}\n")
    for k in missed:   print(f"  MISSED  : {shorten(in_scope[k]['title'], 58)}")
    for k in spurious: print(f"  SPURIOUS: {shorten(got_by_cite[k]['title'], 58)}")
    for k in found:
        if k not in same_req:
            print(f"  DIFFERENT REQUIREMENTS: {in_scope[k]['title']}")
            print(f"      gold : {norm_conditions(in_scope[k]['logic']['requires'])}")
            print(f"      model: {norm_conditions(matched[k]['logic']['requires'])}")
    EXTRACTION_SCORE = {"gold": len(in_scope), "found": len(found), "same_req": len(same_req),
                        "exact": len(same_all), "missed": len(missed), "spurious": len(spurious)}
else:
    print("no model output to score. run with an API key to populate this.")
    EXTRACTION_SCORE = None

gold rules in the extracted section: 7 of 12 document-wide

  matched by identical logic despite a different citation:
     gold  cites "billing more than 1 uos per 30 days of code k0553 will [...]"
     model cites "only one (1) uos of code k0553 may be billed to the dme [...]"

found by extractor : 7/7
  same requirements: 6
  fully identical  : 1  (also matching applies_when and on_fail)
missed             : 0
spurious           : 0

  DIFFERENT REQUIREMENTS: Insulin treatment threshold
      gold : ['csii_pump_medicare_covered|is_true|null', 'insulin_injections_per_day|>=|3']
      model: ['insulin_injections_per_day|>=|3', 'insulin_treated|is_true|null']


### A gate, not a vibe

Here's the uncomfortable part. Sections 8 through 10 are about detecting policy drift
and pricing what it costs in claim decisions. Those results only mean anything if the
ruleset going in is actually correct.

On the first real run it wasn't. The extractor produced two rules that passed
validation and were still nonsense, and running the impact analysis on them turned
twelve claims into manual review and reported zero dollars recovered. The demo looked
broken because it was.

So the honest move is a gate. Score the extraction, and if it doesn't clear the bar,
carry the hand-authored reference ruleset forward instead and say so out loud. That
isn't cheating as long as it's stated plainly — the downstream sections are testing the
drift detector, not the extractor, and feeding them a known-bad ruleset would measure
nothing at all.

A production system would do the same thing, except the gate would be a human reviewer
approving rules before they get promoted. That's layer 7 in the architecture.

In [20]:
GATE_REASONS = []
if not RULES_ARE_MODEL_OUTPUT:
    GATE_REASONS.append("no model output available")
else:
    e = EXTRACTION_SCORE
    if e["spurious"]:
        GATE_REASONS.append(f"{e['spurious']} spurious rule(s) not present in the gold set")
    if e["found"] < e["gold"]:
        GATE_REASONS.append(f"only found {e['found']} of {e['gold']} in-scope gold rules")
    if e["same_req"] < e["found"]:
        GATE_REASONS.append(f"{e['found'] - e['same_req']} rule(s) encode different requirements than gold")

if GATE_REASONS:
    print("EXTRACTION GATE: FAILED")
    for r in GATE_REASONS:
        print(f"  - {r}")
    print("\nCarrying the hand-authored reference ruleset into sections 8-10 instead.")
    print("Everything below therefore measures the DRIFT DETECTOR, not the extractor.")
    print("The extractor's own score is reported above and repeated in the scorecard.")
    working_rules = reference
    RULESET_SOURCE = "hand-authored reference (extraction gate failed)"
else:
    print("EXTRACTION GATE: PASSED. Using model-extracted rules downstream.")
    RULESET_SOURCE = "model-extracted"

print(f"\nruleset going into drift detection: {len(working_rules)} rules, {RULESET_SOURCE}")

EXTRACTION GATE: FAILED
  - 1 rule(s) encode different requirements than gold

Carrying the hand-authored reference ruleset into sections 8-10 instead.
Everything below therefore measures the DRIFT DETECTOR, not the extractor.
The extractor's own score is reported above and repeated in the scorecard.

ruleset going into drift detection: 12 rules, hand-authored reference (extraction gate failed)


## 8. Drift detection

Probably the single most important design decision in this notebook.

The obvious way to detect change is to extract from v1, extract from v2, and diff the
two rulesets. Don't do this. Section 7 already showed why — the extractor isn't
perfectly stable, so re-extracting from a near-identical document produces differences
that come from your own nondeterminism, not from the policy. You'd end up shipping a
change detector whose false positives are self-inflicted.

Instead, hold the ruleset fixed and ask one narrow question at a time:

> Given this revised policy, is this specific rule still supported?

Classification is a lot easier to get right than open-ended extraction, and the output
is a verdict per rule instead of a diff you have to go interpret yourself.

### The four verdicts

| Verdict | Meaning | Action |
|---|---|---|
| `SUPPORTED` | v2 still states this requirement | keep |
| `MODIFIED` | still there, threshold or wording changed | revise |
| `CONTRADICTED` | v2 would now decide differently | retire |
| `UNADDRESSED` | v2 is silent on the subject | human review |

One judgment call is worth stating plainly. When a criterion gets deleted from an
exhaustive "all of the following" list, is that `CONTRADICTED` or `UNADDRESSED`? The
policy never says "testing frequency no longer matters," it just stops mentioning it.
I call that `CONTRADICTED`, because what actually matters is the practical consequence
— the rule now denies claims the policy covers. Silence inside a closed list is still a
decision. That definition is baked into the prompt rather than left for the model to
guess at.

In [21]:
# the model isn't asked for a verdict at all -- it supplies three observations, in
# order, and derive_verdict() below turns them into a verdict from a plain decision
# table. asking for the verdict directly missed real changes ("injections" ->
# "administrations"); asking for a raw diff first flagged cosmetic renumbering as a
# real one. splitting fact from judgement fixed both.
DRIFT_SCHEMA = {
    "type": "object",
    "properties": {
        "evidence_sentence":           {"type": "string"},
        "wording_differences":         {"type": "string"},
        "requirement_still_present":   {"type": "boolean"},
        "was_item_in_criteria_list":   {"type": "boolean"},
        "changes_what_claims_qualify": {"type": "boolean"},
        "explanation":                 {"type": "string"}},
    "required": ["evidence_sentence", "wording_differences", "requirement_still_present",
                 "was_item_in_criteria_list", "changes_what_claims_qualify", "explanation"],
    "additionalProperties": False,
}

def derive_verdict(obs: dict) -> tuple[str, str]:
    """Decision table. Deterministic, inspectable, and identical every time."""
    if not obs["requirement_still_present"]:
        # Gone. Removal from a closed list of criteria contradicts a rule that enforces it,
        # because the rule now denies claims the policy covers.
        return ("CONTRADICTED", "RETIRE") if obs["was_item_in_criteria_list"] else ("UNADDRESSED", "HUMAN_REVIEW")
    if obs["changes_what_claims_qualify"]:
        return ("MODIFIED", "REVISE")
    return ("SUPPORTED", "KEEP")

DRIFT_SYSTEM = """You audit whether an existing claim rule is still supported by a revised policy.

You get one rule, its executable logic, the sentence it was derived from, and the full text
of the revised policy.

You do NOT decide a verdict. You report five observations and something else computes the
verdict from them.

THE QUESTION THROUGHOUT: does this RULE, as its logic is actually written, still do what
the revised policy requires? You are auditing a rule, not proofreading prose. A sentence
that was edited in a way that does not touch what the rule tests is not a change here.

STEP 1 - evidence_sentence. Find the sentence in the REVISED policy corresponding to the one
the rule came from. Copy it VERBATIM, including any leading list number. It is checked by
exact search and an unfindable quote invalidates the finding. If the requirement was deleted
outright, quote instead the sentence that introduces the list it used to belong to.

STEP 2 - wording_differences. Put the two sentences side by side and compare word by word.
List every difference, or "none". Watch the operative noun or verb ("injections" vs
"administrations"), qualifiers that widen or narrow scope ("Medicare-covered", "multiple"),
and any numbers, counts, frequencies or time windows.

STEP 3 - requirement_still_present. Does the revised policy still impose this requirement
anywhere, in any wording? True if it survives even in altered form. False if it is gone.

STEP 4 - was_item_in_criteria_list. Was this requirement one numbered item in a list the
policy says must ALL be met for coverage? This matters only when the requirement is gone,
because deleting an item from a closed list means claims the rule denies are now covered.

STEP 5 - changes_what_claims_qualify. Look at the rule's LOGIC, not its prose. Would a claim
with identical patient and billing facts get a different decision out of that logic under
the revised policy? Answer true only if the logic itself is now wrong.

Answer FALSE when the sentence was edited but the logic still implements exactly what the
policy requires. These never change any claim outcome:
  - a cross-reference into a renumbered list, "criteria (1-4)" becoming "criteria (1-3)"
    because an earlier item was removed. The rule does not test the other criteria. It
    tests its own condition, and that condition is untouched.
  - renumbering of the criteria themselves, item 5 becoming item 4
  - line breaks, hyphenation, spacing, capitalization, punctuation
A long list of differences of that kind is still no change at all."""


def check_drift(rule, new_text, tag=""):
    user = (f"EXISTING RULE\n  id: {rule['rule_id']}\n  title: {rule['title']}\n"
            f"  derived from: \"{rule['source_sentence']}\"\n"
            f"  logic: {json.dumps(rule['logic'])}\n\n"
            f"REVISED POLICY\n---\n{new_text}\n---")
    parsed, src = chat_json(DRIFT_SYSTEM, user, DRIFT_SCHEMA, "drift", run_tag=tag)

    verdict, action = derive_verdict(parsed)          # decision table, not the model
    sp = locate(new_text, parsed["evidence_sentence"])
    parsed.update({"verdict": verdict, "recommended_action": action,
                   "evidence_span": list(sp) if sp else None, "verified": sp is not None,
                   "rule_id": rule["rule_id"]})
    if not parsed["verified"]:              # unverifiable evidence never auto-acts
        parsed["recommended_action"] = "HUMAN_REVIEW"
    return parsed, src


drift = []
try:
    for r in working_rules:
        v, _ = check_drift(r, v2_text, tag=f"2020to2021-{r['rule_id']}")
        drift.append(v)
    DRIFT_IS_MODEL_OUTPUT = True
    print("model reports facts (left), python derives the verdict (right)\n")
    print(f"{'rule':<10} {'present?':<9} {'in list?':<9} {'affects':<8} | {'VERDICT':<14} {'action':<13} {'cite':<5}")
    print("-" * 92)
    for v in drift:
        print(f"{v['rule_id']:<10} {str(v['requirement_still_present']):<9} "
              f"{str(v['was_item_in_criteria_list']):<9} {str(v['changes_what_claims_qualify']):<8} | "
              f"{v['verdict']:<14} {v['recommended_action']:<13} {str(v['verified']):<5}")
    print()
    for v in drift:
        if v["verdict"] != "SUPPORTED":
            print(f"{v['rule_id']}: {shorten(v['wording_differences'], 96)}")
except NoAPIKey as e:
    DRIFT_IS_MODEL_OUTPUT = False
    print("SKIPPED:", e)
    print("\n" + "!" * 74)
    print("!! Using EXPECTED verdicts from gold_revision_history.json so sections 9 and")
    print("!! 10 execute. These are NOT detector output. Do not quote any number below.")
    print("!" * 74)
    _g = json.loads((DATA / "gold_revision_history.json").read_text())
    _act = {"CONTRADICTED": "RETIRE", "MODIFIED": "REVISE", "SUPPORTED": "KEEP", "UNADDRESSED": "HUMAN_REVIEW"}
    _exp = {re.sub(r"\s+", " ", c["affects_source_sentence_v1"]).strip().lower():
            (c["expected_verdict"], _act[c["expected_verdict"]])
            for c in _g["documented_changes"] if c["scoreable"]}
    drift = [{"rule_id": r["rule_id"],
              "verdict": _exp.get(cite_key(r), ("SUPPORTED", "KEEP"))[0],
              "recommended_action": _exp.get(cite_key(r), ("SUPPORTED", "KEEP"))[1],
              "evidence_sentence": "(stand-in, not model output)",
              "explanation": "(stand-in, not model output)",
              "wording_differences": "(stand-in)", "changes_what_claims_qualify": None,
              "requirement_still_present": None, "was_item_in_criteria_list": None,
              "evidence_span": None, "verified": False} for r in working_rules]

model reports facts (left), python derives the verdict (right)

rule       present?  in list?  affects  | VERDICT        action        cite 
--------------------------------------------------------------------------------------------
CGM-001    True      True      False    | SUPPORTED      KEEP          True 
CGM-002    False     True      True     | CONTRADICTED   RETIRE        True 
CGM-003    True      True      True     | MODIFIED       REVISE        True 
CGM-004    True      True      False    | SUPPORTED      KEEP          True 
CGM-005    True      True      False    | SUPPORTED      KEEP          True 
CGM-006    True      True      False    | SUPPORTED      KEEP          True 
SUP-001    True      False     False    | SUPPORTED      KEEP          True 
BGM-001    True      True      False    | SUPPORTED      KEEP          True 
BGM-002    True      True      False    | SUPPORTED      KEEP          True 
BGM-003    True      False     False    | SUPPORTED      KEEP          Tr

In [22]:
# ---- negative test, on real documents --------------------------------------
# CMS published two consecutive 2024 versions where the only change was a HCPCS long
# descriptor, outside the coverage criteria entirely. Better control than a synthetic
# perturbation, since nothing a coverage rule depends on actually moved.

A = coverage_text(PDFS / "LCD - Glucose Monitors (L33822) superseded.pdf")   # eff 04/01/2024
B = coverage_text(PDFS / "LCD - Glucose Monitors (L33822).pdf")              # eff 10/01/2024

from pipeline.pdf_ingest import canonical

tier0 = hashlib.sha256(A.encode()).hexdigest() == hashlib.sha256(B.encode()).hexdigest()
flat_a, flat_b = re.sub(r"\s+", " ", A).strip(), re.sub(r"\s+", " ", B).strip()
tier1 = flat_a == flat_b
tier1b = canonical(A) == canonical(B)

print(f"TIER 0   raw bytes identical?              {tier0}")
print(f"TIER 1   identical after collapsing runs?  {tier1}")
print(f"TIER 1b  identical after removing all ws?  {tier1b}")
print(f"         ({len(A):,} vs {len(B):,} chars, {abs(len(A)-len(B))}-char difference)")
print()

if not tier1:
    sm = __import__("difflib").SequenceMatcher(None, flat_a, flat_b, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            print(f"  the one difference tier 1 sees: {tag} {flat_a[i1:i2]!r} -> {flat_b[j1:j2]!r}")
            print(f"  in context: ...{flat_a[max(0,i1-46):i2+34]}...")

print()
if not tier0 and not tier1 and tier1b:
    print("This is the triage cascade on real data, and the middle step is the lesson.")
    print("  Tier 0 says the bytes changed. True, but useless: the PDF re-typeset itself.")
    print("  Tier 1 collapses whitespace runs and STILL sees a difference, because the")
    print("     newer PDF broke a line inside '(1)-(2)' and left '(1)- (2)'. That stray")
    print("     space is indistinguishable from a real edit under collapse-only rules.")
    print("  Tier 1b removes whitespace entirely and the two are byte-identical.")
    print("     Nothing substantive changed. Stop. ZERO model calls spent.")
    print()
    print("Naive normalization was not enough. Had we escalated on the tier 1 result we")
    print("would have paid a model to look at a document that did not change. At five")
    print("million documents a day that difference is the whole cost model.")
    NEGATIVE_TEST = "passed at tier 1b, no model call needed"
elif tier1b:
    NEGATIVE_TEST = "passed at tier 1b"
else:
    NEGATIVE_TEST = "coverage text genuinely differs, escalation would be correct"
    print("Coverage text genuinely differs. Escalating to the model would be correct here.")

TIER 0   raw bytes identical?              False
TIER 1   identical after collapsing runs?  False
TIER 1b  identical after removing all ws?  True
         (15,302 vs 15,303 chars, 1-char difference)



  the one difference tier 1 sees: insert '' -> ' '
  in context: ... supplies, if the basic coverage criteria (1)-(2) are not met, the item(s) will ...

This is the triage cascade on real data, and the middle step is the lesson.
  Tier 0 says the bytes changed. True, but useless: the PDF re-typeset itself.
  Tier 1 collapses whitespace runs and STILL sees a difference, because the
     newer PDF broke a line inside '(1)-(2)' and left '(1)- (2)'. That stray
     space is indistinguishable from a real edit under collapse-only rules.
  Tier 1b removes whitespace entirely and the two are byte-identical.
     Nothing substantive changed. Stop. ZERO model calls spent.

Naive normalization was not enough. Had we escalated on the tier 1 result we
would have paid a model to look at a document that did not change. At five
million documents a day that difference is the whole cost model.


This matters more than it looks like it should. A change detector that fires on
documents that didn't change is worse than useless, because it buries the real findings
in noise.

The interesting part is in the middle tier, and it's the kind of thing you only find by
running on real documents. Hashing fails, because the PDF genuinely re-typeset itself.
Collapsing whitespace runs also fails, because the newer PDF wrapped a line inside
`(1)-(2)` and left a stray space behind. Only removing whitespace outright shows that
the coverage criteria are byte-for-byte identical.

If the cascade had stopped at tier 1, we'd have paid a model to look at a document that
didn't change. One document doesn't matter. Five million a day is the entire cost
model.

This is also why `canonical()` exists as a separate function from `normalize()` in
`pdf_ingest.py` — one is for comparison, the other for display, and mixing them up is
how you end up with either false positives or broken citation offsets.

In [23]:
# extra check: make the model look at the pair anyway and confirm it also finds
# nothing. costs a few calls, turns "the cheap check said no" into "both agree".
try:
    a_sections = segment(A)
    a_cgm = next(s for s in a_sections if s["name"].startswith("CONTINUOUS GLUCOSE"))
    neg_rules, _ = extract_rules(a_cgm["text"], A, "L33822-2024-04", run_tag="negctl")
    neg_rules = [r for r in neg_rules if not validate_rule(r, A, ALLOWED_FIELDS)[1]]
    noise = [check_drift(r, B, tag=f"neg-{r['rule_id']}")[0] for r in neg_rules]
    flagged = [v for v in noise if v["verdict"] != "SUPPORTED"]
    print(f"rules extracted from the 04/2024 version : {len(neg_rules)}")
    print(f"findings against the 10/2024 version     : {len(flagged)}")
    for v in flagged:
        print(f"   FALSE POSITIVE  {v['rule_id']}: {v['verdict']} - {shorten(v['explanation'], 58)}")
    if not flagged:
        print("   clean. no false positives on a pair that did not substantively change.")
    COSMETIC_FALSE_POSITIVES = len(flagged)
except NoAPIKey:
    COSMETIC_FALSE_POSITIVES = None
    print("skipped, needs an API key")

rules extracted from the 04/2024 version : 6
findings against the 10/2024 version     : 0
   clean. no false positives on a pair that did not substantively change.


In [24]:
# ---- gap sweep: requirements in v2 that no existing rule covers ------------
GAP_SCHEMA = {"type": "object", "properties": {"gaps": {"type": "array", "items": {
    "type": "object",
    "properties": {"requirement": {"type": "string"}, "source_sentence": {"type": "string"},
                   "why_uncovered": {"type": "string"}},
    "required": ["requirement", "source_sentence", "why_uncovered"], "additionalProperties": False}}},
    "required": ["gaps"], "additionalProperties": False}

GAP_SYSTEM = """You find coverage requirements in a policy that an existing ruleset does not cover.

You get the revised policy and the list of rules that already exist. Identify checkable
requirements the policy states that no listed rule addresses. Quote the source sentence
VERBATIM. If every requirement is already covered, return an empty list. An empty list is
the correct answer more often than not, so do not invent gaps."""

try:
    inventory = "\n".join(f"- {r['rule_id']}: {r['summary']}" for r in working_rules)
    gaps, _ = chat_json(GAP_SYSTEM, f"EXISTING RULES\n{inventory}\n\nREVISED POLICY\n---\n{v2_text}\n---",
                        GAP_SCHEMA, "gaps", run_tag="v2gap")
    print(f"gaps found: {len(gaps['gaps'])}")
    for g in gaps["gaps"]:
        ok = locate(v2_text, g["source_sentence"]) is not None
        print(f"  [{'ok ' if ok else 'BAD'}] {g['requirement']}")
        print(f"        {shorten(g['why_uncovered'], 74)}")
    if not gaps["gaps"]:
        print("  none. every requirement in v2 is covered by an existing rule.")
    N_GAPS = len(gaps["gaps"])
except NoAPIKey:
    N_GAPS = None
    print("skipped, needs an API key")

gaps found: 13
  [ok ] More than one spring powered device (code A4258) per 6 months is not reasonable and necessary.
        No existing rule addresses a utilization limit for spring powered [...]
  [ok ] The medical necessity for a laser skin piercing device (code E0620) and related lens shield cartridge (code A4257) has not been established; therefore, claims for code E0620 and/or code A4257 will be denied as not reasonable and necessary.
        No existing rule addresses denial of coverage for E0620 or A4257.
  [ok ] For a beneficiary who is not currently being treated with insulin administrations, up to 100 test strips and up to 100 lancets every 3 months are covered if the basic coverage criteria (1)-(2) (above) are met.
        No existing rule specifies quantity limits for test strips and [...]
  [ok ] For a beneficiary who is currently being treated with insulin administrations, up to 300 test strips and up to 300 lancets every 3 months are covered if basic coverage criteria 

I predicted zero gaps here, and I was wrong. The detector found two, and both
are real: the Standard Written Order requirement and the Proof of Delivery
requirement.

Neither is new in the 07/18/2021 revision. They were always in the policy, in the
GENERAL section, which section 5 never extracted from in the first place. So these are
genuine coverage requirements the ruleset doesn't enforce, which is exactly what a GAP
is supposed to catch. The detector was right and my expectation was wrong.

That distinction is worth holding onto. A gap can show up two ways: the policy added a
requirement, or your extraction never covered part of the document to begin with. Both
are real exposure. The second is probably more common in practice, since nobody
extracts every section of every policy on the first pass.

(The later 04/16/2023 revision does add a genuinely new criterion about sufficient
training, which would exercise the first kind of gap. That's the natural next document
pair to load if this gets extended.)

## 9. What it costs, in claim decisions

Verdicts are abstract. Money isn't. So we run the same twenty claims twice — once
under the original ruleset, once under the corrected one.

### The tool doesn't fix anything by itself

This is the difference between a demo and something you could actually deploy.

- `RETIRE` gets applied automatically. Dropping a rule the policy no longer supports is
  safe, since it can only stop denials, never create new ones.
- `REVISE` does not get applied automatically. The tool flags that the insulin
  criterion is now too narrow, but a human decides what the new logic should be. That
  edit is written out explicitly below so you can see exactly where the automation
  stops.

The system triages, but a person still authors the fix.

In [25]:
retire = {v["rule_id"] for v in drift if v["recommended_action"] == "RETIRE"}
revise = {v["rule_id"] for v in drift if v["recommended_action"] == "REVISE"}
print("flagged RETIRE:", sorted(retire) or "none")
print("flagged REVISE:", sorted(revise) or "none")

# deepcopy matters. the human patch edits rules in place, and without a copy it would
# reach back through shared dicts and rewrite the very baseline we compare against.
corrected = [copy.deepcopy(r) for r in working_rules if r["rule_id"] not in retire]

# The human step. Keyed on the cited sentence rather than the rule id, because a
# model-extracted ruleset invents its own ids and the citation is the stable identifier.
HUMAN_PATCH = {
    "the beneficiary is insulin-treated with multiple (three or more) daily injections of insulin "
    "or a medicare-covered continuous subcutaneous insulin infusion (csii) pump; and,": {
        "source_sentence": "The beneficiary is insulin-treated with multiple (three or more) daily "
                           "administrations of insulin or a continuous subcutaneous insulin infusion "
                           "(CSII) pump; and,",
        "requires": [{"field": "insulin_administrations_per_day", "op": ">=", "value": 3},
                     {"field": "csii_pump", "op": "is_true", "value": None}],
    }
}

for r in corrected:
    patch = HUMAN_PATCH.get(cite_key(r))
    if patch and r["rule_id"] in revise:
        r["logic"]["requires"] = patch["requires"]
        r["source_sentence"]   = patch["source_sentence"]
        sp = locate(v2_text, patch["source_sentence"])
        r["source_span"], r["source_doc"] = (list(sp) if sp else None), "L33822-2021"
        assert sp, "the patched citation must exist in v2"
        print(f"\nhuman patch applied to {r['rule_id']}:")
        print("   injections -> administrations, and 'Medicare-covered' dropped from the pump test")

print(f"\nruleset: {len(working_rules)} rules before, {len(corrected)} after")

flagged RETIRE: ['CGM-002']
flagged REVISE: ['CGM-003']

human patch applied to CGM-003:
   injections -> administrations, and 'Medicare-covered' dropped from the pump test

ruleset: 12 rules before, 11 after


In [26]:
before  = {c["claim_id"]: adjudicate(c, working_rules) for c in claims}
after   = {c["claim_id"]: adjudicate(c, corrected)     for c in claims}
amounts = {c["claim_id"]: c["billed_amount"] for c in claims}

def tally(res):
    out = {"PAY": 0, "DENY": 0, "REVIEW": 0}
    for r in res.values():
        out[r["decision"]] += 1
    return out

b, a = tally(before), tally(after)
print(f"{'':10} {'PAY':>5} {'DENY':>5} {'REVIEW':>7}")
print(f"{'before':<10} {b['PAY']:>5} {b['DENY']:>5} {b['REVIEW']:>7}")
print(f"{'after':<10} {a['PAY']:>5} {a['DENY']:>5} {a['REVIEW']:>7}\n")

flips = [(cid, before[cid]["decision"], after[cid]["decision"])
         for cid in before if before[cid]["decision"] != after[cid]["decision"]]

print(f"{'claim':<8} {'was':<8} {'now':<8} {'amount':>10}   caused by")
print("-" * 92)
recovered = 0.0
for cid, was, now in sorted(flips):
    if was == "DENY" and now == "PAY":
        recovered += amounts[cid]
    gone = [f["rule_id"] for f in before[cid]["fired"] if f["rule_id"] in retire | revise]
    print(f"{cid:<8} {was:<8} {now:<8} {amounts[cid]:>10,.2f}   {', '.join(gone) or 'ruleset changed'}")
print("-" * 92)
print(f"{len(flips)} of {len(claims)} claims changed decision")
print(f"${recovered:,.2f} was being wrongly withheld across the DENY to PAY flips")

             PAY  DENY  REVIEW
before         7    10       3
after         13     5       2

claim    was      now          amount   caused by
--------------------------------------------------------------------------------------------
C-003    DENY     PAY          237.29   CGM-002
C-004    DENY     PAY          259.55   CGM-002
C-007    DENY     PAY          237.29   CGM-002
C-008    REVIEW   PAY          237.29   CGM-002
C-011    DENY     PAY          237.29   CGM-003
C-016    DENY     PAY          237.29   CGM-003
--------------------------------------------------------------------------------------------
6 of 20 claims changed decision
$1,208.71 was being wrongly withheld across the DENY to PAY flips


## 10. Scorecard

Every number, including the ones that don't flatter the tool.

In [27]:
gold      = json.loads((DATA / "gold_revision_history.json").read_text())
scoreable = [c for c in gold["documented_changes"] if c["scoreable"]]
excluded  = gold.get("excluded_from_scoring", [])
must_not  = gold.get("must_not_flag", [])

print("=" * 78)
print("DRIFT ACCURACY vs the revision history CMS published inside the LCD")
print("=" * 78)
if not DRIFT_IS_MODEL_OUTPUT:
    print("*** STAND-IN DATA. Verdicts below came from the gold file, not the detector.")
    print("*** This measures nothing until re-run with an API key.\n")

by_rule  = {v["rule_id"]: v for v in drift}
cite2rid = {cite_key(r): r["rule_id"] for r in working_rules}
norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()

affected, caught = set(), 0
for ch in scoreable:
    rid = cite2rid.get(norm(ch["affects_source_sentence_v1"]))
    affected.add(rid)
    got = by_rule.get(rid, {}).get("verdict")
    hit = got == ch["expected_verdict"]
    caught += hit
    print(f"  [{'HIT ' if hit else 'MISS'}] {ch['change_id']}: {shorten(ch['revision_text'], 56)}")
    if rid is None:
        print("          no rule in the working set cites this sentence, so nothing could be flagged")
    print(f"          expected {ch['expected_verdict']:<13} got {got}")

print(f"\n  caught: {caught} of {len(scoreable)} documented edits")
print(f"  {gold['scoring_notes']}")

print("\n  MUST-NOT-FLAG cases (real false-positive traps):")
for mn in must_not:
    rid = cite2rid.get(norm(mn["source_sentence_v1"]))
    got = by_rule.get(rid, {}).get("verdict")
    ok = got == mn["expected_verdict"]
    print(f"    [{'PASS' if ok else 'FAIL'}] {mn['case_id']} rule={rid} expected "
          f"{mn['expected_verdict']}, got {got}")
    if not ok:
        print(f"          {shorten(mn['why'], 90)}")

false_alarms = [v for v in drift
                if v["rule_id"] not in affected and v["verdict"] != "SUPPORTED"]
print(f"\n  false alarms: {len(false_alarms)} rules flagged that the revision history does not mention")
for v in false_alarms:
    print(f"      {v['rule_id']}: {v['verdict']}")

for ch in excluded:
    print(f"\n  excluded from scoring: {ch['change_id']} - {shorten(ch['reason'], 92)}")

DRIFT ACCURACY vs the revision history CMS published inside the LCD
  [HIT ] R1: Removed: Four times or more per day testing with [...]
          expected CONTRADICTED  got CONTRADICTED
  [HIT ] R2: Revised: "injections" to "administrations" for [...]
          expected MODIFIED      got MODIFIED
  [HIT ] R3: Removed: "Medicare-covered" from CSII pump [...]
          expected MODIFIED      got MODIFIED

  caught: 3 of 3 documented edits
  R2 and R3 revise the same sentence, so they are not independent observations. One correct MODIFIED verdict on that criterion satisfies both. Report this as 2 affected criteria across 3 documented edits rather than 3 independent detections. With n this small, always print raw counts alongside any percentage.

  MUST-NOT-FLAG cases (real false-positive traps):
    [PASS] N1 rule=CGM-005 expected SUPPORTED, got SUPPORTED

  false alarms: 0 rules flagged that the revision history does not mention

  excluded from scoring: R4 - Coding verification lives in

In [28]:
print("=" * 78); print("EXTRACTION"); print("=" * 78)
print(f"  stability   : " + (f"{STABILITY[0]} of {STABILITY[1]} distinct rules unanimous across {N_RUNS} runs"
                             if STABILITY else "not measured (needs API key)"))
if EXTRACTION_SCORE:
    e = EXTRACTION_SCORE
    print(f"  vs gold     : found {e['found']}/{e['gold']} in-scope rules, "
          f"{e['same_req']} with identical requirements, {e['exact']} identical throughout")
    print(f"                {e['missed']} missed, {e['spurious']} spurious")
    print(f"  gate        : {'PASSED' if not GATE_REASONS else 'FAILED - ' + '; '.join(GATE_REASONS)}")
    print(f"  ruleset used downstream: {RULESET_SOURCE}")
else:
    print("  vs gold     : not measured (needs API key)")

print()
print("=" * 78); print("NEGATIVE CONTROL (real 04/2024 vs 10/2024 pair)"); print("=" * 78)
print(f"  cheap triage : {NEGATIVE_TEST}")
print(f"  model check  : " + (f"{COSMETIC_FALSE_POSITIVES} false positives"
                              if COSMETIC_FALSE_POSITIVES is not None else "not measured (needs API key)"))

print()
print("=" * 78); print("IMPACT"); print("=" * 78)
print(f"  claims evaluated : {len(claims)}  (3 more rejected as malformed at load)")
print(f"  decisions changed: {len(flips)}")
print(f"  wrongly withheld : ${recovered:,.2f}")

print()
print("=" * 78); print("KNOWN LIMITATIONS"); print("=" * 78)
for line in [
    "n is tiny. 3 documented edits across 2 affected criteria, 20 synthetic claims, one",
    "  document pair. These demonstrate behavior; they are not statistically meaningful.",
    "The gold ruleset and the claim set were authored by the same person who built the",
    "  extractor, so this is not an independent evaluation. Only the revision history is",
    "  genuinely external, and that is why it carries the most weight here.",
    "CONTRADICTED vs UNADDRESSED for a deleted criterion is a definition written into the",
    "  prompt, not something the model discovered.",
    "One clinical area (DME glucose monitors), one document format, one payer (Medicare).",
    "The reference ruleset covers the CGM and BGM criteria plus the written-order rule. It",
    "  does not cover test strip and lancet utilization limits, which is why the GAP sweep",
    "  returns a long list. Those gaps are real, not false positives, but they measure how",
    "  incomplete the ruleset is rather than how good the detector is.",
    "No OCR. Every PDF here has an extractable text layer; scanned policy would need it.",
    "Payer-provider contracts are not covered at all, because no public corpus exists and",
    "  fabricating inputs would make the whole demonstration worthless.",
]:
    print(f"  - {line}")

EXTRACTION
  stability   : 7 of 9 distinct rules unanimous across 5 runs
  vs gold     : found 7/7 in-scope rules, 6 with identical requirements, 1 identical throughout
                0 missed, 0 spurious
  gate        : FAILED - 1 rule(s) encode different requirements than gold
  ruleset used downstream: hand-authored reference (extraction gate failed)

NEGATIVE CONTROL (real 04/2024 vs 10/2024 pair)
  cheap triage : passed at tier 1b, no model call needed
  model check  : 0 false positives

IMPACT
  claims evaluated : 20  (3 more rejected as malformed at load)
  decisions changed: 6
  wrongly withheld : $1,208.71

KNOWN LIMITATIONS
  - n is tiny. 3 documented edits across 2 affected criteria, 20 synthetic claims, one
  -   document pair. These demonstrate behavior; they are not statistically meaningful.
  - The gold ruleset and the claim set were authored by the same person who built the
  -   extractor, so this is not an independent evaluation. Only the revision history is
  -   ge

---

# 11. Three things added after the first real run

Sections 1 through 10 are the pipeline as it was first built. Running it against the
real documents turned up problems that weren't visible until there were actual
numbers, so three things got added afterward. They live in `backend/pipeline/` and get
imported here rather than retyped, so this notebook and the running application can't
quietly drift apart. The last cell checks that directly.

| Addition | Fixes |
|---|---|
| Self-consistency voting | five extraction runs were being made and four thrown away |
| Round-trip verification | scoring needed a gold ruleset, which real documents never have |
| Provider explanations | the summarization focus area had no output at all |

In [29]:
from pipeline.rules import vote_rules, requires_hash as rq_hash
from pipeline import roundtrip, explain as explain_mod

print("imported from backend/pipeline/, the same code the API and CLI run")
print("stability runs available:", len(runs))

imported from backend/pipeline/, the same code the API and CLI run
stability runs available: 5


## 11.1 Self-consistency voting, and why it wasn't enough on its own

Section 7 already makes five extraction runs to measure stability, then only uses one
of them. The other four were already paid for, so they may as well vote — identity is
the cited sentence, and among the runs that produced a rule for a given sentence, the
most common logic wins.

It didn't fix the broken rule, and that's honestly the interesting part.

In [30]:
voted, report = vote_rules(runs)
print(f"single run (section 5): {len(extracted)} rules   after voting: {len(voted)} rules\n")

for v in report:
    if v.get("disputed"):
        print(f"DISPUTED  {shorten(v['title'], 60)}")
        for alt in v["alternatives"]:
            print(f"   {alt['votes']} vote(s)  {alt['requires']}")
        print()
    elif not v["kept"]:
        print(f"DROPPED   {v['reason']}")

single run (section 5): 7 rules   after voting: 8 rules

DISPUTED  Insulin treatment with multiple daily injections or [...]
   2 vote(s)  ['insulin_injections_per_day|>=|3', 'insulin_treated|is_true|null']
   1 vote(s)  ['insulin_injections_per_day|>=|3']
   1 vote(s)  ['csii_pump_medicare_covered|is_true|null']
   1 vote(s)  ['csii_pump_medicare_covered|is_true|null', 'insulin_injections_per_day|>=|3']
   1 vote(s)  ['insulin_injections_per_day|>=|3']

DISPUTED  BGM and supplies not covered with approved CGM and supplies
   3 vote(s)  ['device_type|!=|"therapeutic_cgm"']
   2 vote(s)  ['procedure_code|!=|"K0554"']

DROPPED   appeared in 1/5 runs, below the 3-run threshold


What you should see: the insulin criterion comes back with several different
encodings across the five runs, and the plurality is the one that drops the "or a
Medicare-covered CSII pump" alternative.

Which is the actual lesson here. Majority voting corrects random variance, not
systematic bias. The model wasn't guessing differently each time — it was wrong the
same way more often than it was right. A vote over a biased sampler just returns the
bias.

So voting by itself isn't the answer. It needs a referee that knows something the vote
doesn't.

## 11.2 Round-trip verification

Compile the policy into a rule, then decompile that rule back into prose and compare
it against the original sentence.

The reason to do this instead of just scoring against `reference_ruleset.json` is
coverage of the real world. Scoring against a hand-authored gold set only works on
documents somebody has already done by hand, which is basically none of the documents
anyone actually cares about. Here the source sentence is its own ground truth, so the
check works anywhere.

The decompile step deliberately never sees the source sentence. If it did, it would
just paraphrase the policy back instead of describing the logic, and the comparison
would agree with itself every single time.

In [31]:
rule = next(r for r in voted if "insulin" in r["title"].lower())

print("RULE LOGIC the model was shown (no policy text):")
print(json.dumps(rule["logic"]["requires"], indent=2))

finding = roundtrip.check(rule, tag=f"nb-{rule['rule_id']}")
print(f"\nDECOMPILED BACK TO ENGLISH:\n  {finding['decompiled']}")
print(f"\nORIGINAL POLICY SENTENCE:\n  {shorten(finding['source_sentence'], 150)}")
print(f"\nfaithful : {finding['faithful']}")
print(f"severity : {finding['severity']}")
print(f"missing  : {finding['missing_from_rule']}")

RULE LOGIC the model was shown (no policy text):
[
  {
    "field": "insulin_injections_per_day",
    "op": ">=",
    "value": 3
  },
  {
    "field": "insulin_treated",
    "op": "is_true",
    "value": true
  }
]

DECOMPILED BACK TO ENGLISH:
  If device_type is equal to therapeutic_cgm, then either insulin_injections_per_day is greater than or equal to 3, or insulin_treated is true.

ORIGINAL POLICY SENTENCE:
  3. The beneficiary is insulin-treated with multiple (three or more) daily injections of insulin or a Medicare-covered continuous subcutaneous [...]

faithful : False
severity : material
missing  : The rule omits the requirement that, for CSII pump users, the pump must be Medicare-covered (csii_pump_medicare_covered). The rule also allows insulin_treated alone to qualify, without requiring either three or more daily injections or use of a Medicare-covered CSII pump, which is broader than the policy.


What you should see: the decompiled rendering describes a test on injection
counts, the policy sentence offers an alternative route to qualifying through a pump,
and the comparison correctly flags that alternative as a material omission.

No gold ruleset got consulted anywhere in that cell.

### Getting the first version of this wrong

The first attempt flagged eight rules out of eight as material omissions, including
ones that were completely correct. It was checking prose completeness instead of claim
consequence, so a rule testing `in_person_visit_within_6mo` got marked deficient for
not also restating "to evaluate their diabetes control."

Same mistake as the first drift prompt, basically. The fix was to say plainly that a
claim field stands in for a whole clause, spell out what actually counts as material
(a dropped alternative, a changed threshold, a dropped qualifier) versus what doesn't
(narrative context, pointers to code lists, anything the schema simply can't express),
and show the model which fields exist so it knows what could have been encoded in the
first place.

## 11.3 The combination is what actually worked

Voting knows which encoding is most popular. Round-trip knows which encoding is
faithful. Neither one is enough by itself, but together the vote proposes and the
verifier checks it: when runs disagree, check each variant against its source and
prefer whichever one comes back faithful. Votes only break ties among candidates that
are equally faithful.

In [32]:
def arbiter(candidate):
    return roundtrip.check(candidate, tag=f"arb-{rq_hash(candidate)}")["severity"]

arbitrated, report2 = vote_rules(runs, arbiter=arbiter)

for v in report2:
    if v.get("disputed"):
        print(f"{shorten(v['title'], 58)}   arbitrated={v.get('arbitrated')}")
        for alt in v["alternatives"]:
            mark = "  <-- WON" if alt["won"] else ""
            print(f"   {alt['votes']} vote(s)  roundtrip={str(alt['roundtrip']):<9} {alt['requires']}{mark}")
        print()

# same two-pass matching the app uses: cited sentence first, then identical logic
# for the leftovers, since a paragraph can state one requirement twice.
gold_by_cite = {cite_key(r): r for r in reference}
in_scope = {k: r for k, r in gold_by_cite.items()
            if locate(v1_cgm["text"], r["source_sentence"])}
got = {cite_key(r): r for r in arbitrated}

matched, used = {}, set()
for k in in_scope:
    if k in got:
        matched[k], _ = got[k], used.add(k)
for k, gold_rule in in_scope.items():
    if k in matched:
        continue
    for gk, got_rule in got.items():
        if gk not in used and rq_hash(got_rule) == rq_hash(gold_rule):
            matched[k], _ = got_rule, used.add(gk)
            break

same = sum(1 for k in matched if rq_hash(matched[k]) == rq_hash(in_scope[k]))
print(f"rules whose requirements match the hand-authored gold: {same}/{len(in_scope)}")

Insulin treatment with multiple daily injections or [...]   arbitrated=True
   2 vote(s)  roundtrip=material  ['insulin_injections_per_day|>=|3', 'insulin_treated|is_true|null']
   1 vote(s)  roundtrip=material  ['insulin_injections_per_day|>=|3']
   1 vote(s)  roundtrip=material  ['csii_pump_medicare_covered|is_true|null']
   1 vote(s)  roundtrip=none      ['csii_pump_medicare_covered|is_true|null', 'insulin_injections_per_day|>=|3']  <-- WON
   1 vote(s)  roundtrip=material  ['insulin_injections_per_day|>=|3']

BGM and supplies not covered with approved CGM and [...]   arbitrated=False
   3 vote(s)  roundtrip=material  ['device_type|!=|"therapeutic_cgm"']  <-- WON
   2 vote(s)  roundtrip=material  ['procedure_code|!=|"K0554"']

rules whose requirements match the hand-authored gold: 7/7


What you should see: the winning encoding is the one with a single vote. The
two-vote plurality was material, the one-vote minority was faithful, and the verifier
picked the minority.

Agreement with the hand-authored ruleset goes from six of seven to seven of seven.

Worth being precise about what that does and doesn't show, though. This is one
disputed rule in one document. What it demonstrates is a technique — ensemble plus
verifier re-ranking — not a measured accuracy gain you could generalize from.

## 11.4 Provider explanations

This is the summarization focus area, and it's also the thing that makes a denial
survivable in the real world. A provider who gets a denial is entitled to know which
requirement failed and where it comes from, and "edit 4471 fired" isn't an answer to
that.

The model writes the prose and decides nothing. The decision, the rule that fired, and
the citation are all handed to it directly, and the prompt forbids introducing any
requirement that isn't already in the trace.

In [33]:
claim = by_id["C-007"]
result = before["C-007"]

out = explain_mod.explain(claim, result, tag="nb-C-007")
print(f"DECISION : {out['decision']}\n")
print(f"HEADLINE : {out['headline']}\n")
print("EXPLANATION:")
for line in __import__("textwrap").wrap(out["explanation"], 84):
    print(" ", line)
print("\nNEXT STEP:")
for line in __import__("textwrap").wrap(out["what_to_do"], 84):
    print(" ", line)

DECISION : DENY

HEADLINE : Claim denied: beneficiary did not meet BGM testing frequency requirement.

EXPLANATION:
  This claim is denied because the beneficiary did not meet the requirement that "The
  beneficiary has been using a BGM and performing frequent (four or more times a day)
  testing; and," as stated in policy. The record shows only one blood glucose test per
  day.

NEXT STEP:
  No action needed unless you have documentation showing four or more BGM tests per
  day; if so, submit that with a new claim.


## 11.5 Reconciliation

A notebook and an application that report different numbers are worse than either one
alone, because nobody reading them can tell which one is actually true. So this cell
runs the shipped orchestrator and checks that it agrees with what this notebook just
computed.

If it fails, the two have drifted apart, and the notebook is the one to fix.

In [34]:
from pipeline.run import analyze

app = analyze()
a_adj = app["adjudication"]

checks = [
    ("claims evaluated",       len(claims),                    len(app["claims"])),
    ("decisions changed",      len(flips),                     len(a_adj["flips"])),
    ("wrongly withheld",       round(recovered, 2),            a_adj["recovered"]),
    ("documented changes",     len(scoreable),                 app["scorecard"]["total_changes"]),
    ("reference ruleset size", len(reference),                 len(app["reference_ruleset"])),
]

print(f"{'check':<24} {'notebook':>12} {'application':>14}")
print("-" * 54)
ok = True
for name, nb_val, app_val in checks:
    match = nb_val == app_val
    ok &= match
    print(f"{name:<24} {str(nb_val):>12} {str(app_val):>14}   {'ok' if match else 'MISMATCH'}")

print()
print(f"drift verdicts caught : notebook {caught}/{len(scoreable)}, "
      f"application {app['scorecard']['caught']}/{app['scorecard']['total_changes']}")
assert ok, "notebook and application disagree"
print("\nnotebook and application agree")

check                        notebook    application
------------------------------------------------------
claims evaluated                   20             20   ok
decisions changed                   6              6   ok
wrongly withheld              1208.71        1208.71   ok
documented changes                  3              3   ok
reference ruleset size             12             12   ok

drift verdicts caught : notebook 3/3, application 3/3

notebook and application agree
